# M03-02 — Reglas de negocio

[← Anterior](02-lab-enriquecimiento.ipynb) · [Siguiente →](../M04-integracion-agregacion/01-teoria.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Capar el descuento, normalizar el canal, marcar lo cobrable y persistir `data/staging/fact_lines`.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M03-02-reglas-negocio.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


### Paso 1 — Reconstruye `lines` (1980)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Este notebook es autónomo: repito lectura + join + gmv_line + order_month.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** 1980 filas y 13 GMV negativos (punto de partida).

**Por qué este paso.** Si empiezas “en el aire”, no sabes si el capado funcionó.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col, date_format

spark = get_spark("novashop-m03")
orders = spark.read.parquet(str(STAGING / "orders_clean"))
items = spark.read.parquet(str(STAGING / "order_items_clean"))
lines = (
    items.join(orders, "order_id", "inner")
    .withColumn("gmv_line", col("qty") * col("unit_price") * (1 - col("discount")))
    .withColumn("order_month", date_format(col("order_ts"), "yyyy-MM"))
)
print(lines.count(), lines.where(col("gmv_line") < 0).count())


### Paso 2 — Tres reglas y recalcular GMV

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

least(..., 1) evita GMV negativo. marketplace/WEB/App no sirven para un groupBy. Recalculo gmv_line AL FINAL.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** 1980 filas; `gmv_line < 0` pasa a **0**.

**Por qué este paso.** Si capas el discount *después* de calcular GMV y no recalculas, siguen los 13 negativos.


In [ ]:
from pyspark.sql.functions import when, lower, least, lit

fact = (
    lines.withColumn("discount", least(col("discount"), lit(1.0)))
    .withColumn(
        "channel_norm",
        when(lower(col("channel")).isin("web", "app", "store"), lower(col("channel")))
        .otherwise(lit("other")),
    )
    .withColumn("is_billable", col("status") == "paid")
    .withColumn(
        "gmv_line",
        col("qty") * col("unit_price") * (1 - col("discount")),
    )
)
print("filas", fact.count())
print("gmv < 0", fact.where(col("gmv_line") < 0).count())


### Paso 3 — Valida el dominio

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Un set cerrado se comprueba con groupBy, no a ojo.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Canales solo `app`, `other`, `store`, `web`. `discount > 1` → **0**. Líneas cobrables **1127**.

**Por qué este paso.** El fact guarda las 1980; el flag decide en M04. No filtres `is_billable` al escribir.


In [ ]:
fact.groupBy("channel_norm").count().orderBy("channel_norm").show()
print("discount > 1", fact.where(col("discount") > 1).count())
print("billable", fact.where(col("is_billable")).count())


### Paso 4 — Escribe fact_lines

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

M04 parte de este fact. CSV perdería tipos y el boolean.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** **1980** al releer.

**Por qué este paso.** M04 parte de este fact. CSV perdería tipos y el boolean.

Opcional: `fact.explain("formatted")` y señala el join. Es el puente a M06.


In [ ]:
dest = STAGING / "fact_lines"
fact.write.mode("overwrite").parquet(str(dest))
print(spark.read.parquet(str(dest)).count())


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

`discount <= 1` en todas las filas; `channel_norm` ⊆ {web, app, store, other}; count 1980.
Tres checks en verde en *tu* notebook (cada uno con Markdown).


## Mejora — El plan incluye el join

Lanza `fact.explain("formatted")` y señala (en Markdown) la línea del join / Exchange.

Si te atasca, el código está en la celda siguiente.


In [ ]:
En el plan físico aparece un BroadcastHashJoin o SortMergeJoin con `order_id`. Si no lo ves, estás explicando `lines` *antes* del join.


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| Siguen 13 GMV negativos | No recalculaste gmv_line | Recalcula al final de la cadena |
| channel_norm tiene WEB | Faltó lower | `lower(col("channel"))` antes del isin |
| 1127 no sale | Filtraste is_billable al escribir | El fact guarda 1980 |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M04 — teoría](../M04-integracion-agregacion/01-teoria.ipynb).
